# __FINAL PROJECT PHASE II__

**Project:** Do SEC insider disclosures predict abnormal stock returns?

**Group members:** [Add names]

*Steps 1 to 4 are complete in code and documentation. Step 4 outputs require a fresh run with the saved raw data. EDA, written sections, and the final execution and PDF remain to be completed.*

# __RESEARCH QUESTIONS__

Do publicly disclosed insider purchases and sales predict abnormal stock returns?
Over what horizon does any predictive relationship appear?

Does the relationship differ by insider role, trade size, or whether the trade was
made under a pre-scheduled plan?

We will refine the scope after checking the available sample. An abnormal return
is the difference between a stock's actual return and its expected return given
its historical relationship with the market.

### Importing

We use SciPy, NumPy, DuckDB, and Seaborn. Pandas reads and holds the data tables,
and Matplotlib supports the plots. These imports do not download any data.

In [1]:
# Imports and plot settings
import numpy as np
import pandas as pd
from scipy import stats
import duckdb
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

### Data overview

The local inputs currently include:

- **SEC insider transaction data:** filing metadata, nonderivative transactions,
  reporting owners, and footnotes from all 24 quarters in **2020-2025**.
- **Historical S&P 500 membership:** dated index-membership snapshots, used later
  to select companies that belonged to the index at each disclosure date.

**Individual-company stock prices:** the weekly yfinance download cell below
collects and saves these when run. This replaces the proposal's planned Tiingo
collection. A separate market-benchmark series is still needed for abnormal returns.

**SEC source:** https://www.sec.gov/data-research/sec-markets-data/insider-transactions-data-sets

**Membership source:** https://github.com/fja05680/sp500

**Price-source documentation:** https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html

**Final company sample and retained observation counts:** [Add after cleaning and matching]

**Shared raw-dataset link for reviewers:** [Add project data link]

### Loading the SEC data

Our local SEC data covers **2020 Q1 through 2025 Q4**, in 24 quarter folders under
`data/raw/sec/`. For each quarter, we read `SUBMISSION`, `NONDERIV_TRANS`,
`REPORTINGOWNER`, and `FOOTNOTES` as tab-separated files using `pd.read_csv`.
We combine matching tables with `pd.concat` and add `SOURCE_QUARTER` so every row
can be traced back to its original quarter.

All fields are initially read as strings to preserve identifiers and inspect the
raw values before cleaning. Optional fields present in only some quarters are
retained, with missing values elsewhere. The raw TSV files are unchanged.

Run the cell below once after opening a fresh kernel. It prints progress as each
combined table finishes. The four table types total approximately **1.31 GB** on
disk before loading into memory. Stock-price files will be added separately.

In [2]:
# Load the four SEC tables from all 24 quarters, preserving their source quarter.
from pathlib import Path
from time import perf_counter

sec_dir = Path("data/raw/sec")
if not sec_dir.is_dir():
    sec_dir = Path("insider-trading-phase2/data/raw/sec")  # If opened from the repo root.

quarter_folders = [
    sec_dir / f"{year}q{quarter}_form345"
    for year in range(2020, 2026)
    for quarter in range(1, 5)
]
table_names = ["SUBMISSION", "NONDERIV_TRANS", "REPORTINGOWNER", "FOOTNOTES"]
missing_files = [str(folder / f"{table}.tsv")
                 for folder in quarter_folders for table in table_names
                 if not (folder / f"{table}.tsv").is_file()]
if missing_files:
    raise FileNotFoundError("Check the SEC folder location or missing files:\n" + "\n".join(missing_files))

sec_tables = {}
file_counts = []
load_start = perf_counter()
for table in table_names:
    quarter_frames = []
    for folder in quarter_folders:
        frame = pd.read_csv(folder / f"{table}.tsv", sep="\t", dtype="string")
        frame["SOURCE_QUARTER"] = folder.name.split("_")[0]
        file_counts.append({"quarter": frame["SOURCE_QUARTER"].iloc[0] if len(frame) else folder.name.split("_")[0],
                            "table": table, "rows": len(frame)})
        quarter_frames.append(frame)
    sec_tables[table] = pd.concat(quarter_frames, ignore_index=True)
    del quarter_frames, frame
    print(f"{table}: {len(sec_tables[table]):,} rows loaded", flush=True)

df_submission = sec_tables["SUBMISSION"]
df_transactions = sec_tables["NONDERIV_TRANS"]
df_owners = sec_tables["REPORTINGOWNER"]
df_footnotes = sec_tables["FOOTNOTES"]
df_file_counts = pd.DataFrame(file_counts)
print(f"Loaded {len(file_counts)} files from {len(quarter_folders)} quarters in {perf_counter() - load_start:.1f} seconds.")

SUBMISSION: 1,211,888 rows loaded


NONDERIV_TRANS: 1,841,674 rows loaded


REPORTINGOWNER: 1,323,957 rows loaded


FOOTNOTES: 2,865,764 rows loaded


Loaded 96 files from 24 quarters in 29.6 seconds.


### Initial inspection

The summary below reports the dimensions of each combined table and the row counts
for every quarter. Use these checks to look for empty or incomplete inputs before
cleaning. `df_transactions.head()` displays a few raw transaction rows; these
counts include all nonderivative transaction codes and have not been filtered to P/S.

In [3]:
# Inspect combined table sizes, source-quarter coverage, and raw transaction fields.
from IPython.display import display

summary = pd.DataFrame([
    {"table": name, "rows": len(frame), "columns": frame.shape[1],
     "quarters": frame["SOURCE_QUARTER"].nunique()}
    for name, frame in sec_tables.items()
])
display(summary)
display(df_file_counts.pivot(index="quarter", columns="table", values="rows"))
print("Transaction columns:", df_transactions.columns.tolist())
display(df_transactions.head())

,table,rows,columns,quarters
0,SUBMISSION,1211888,15,24
1,NONDERIV_TRANS,1841674,29,24
2,REPORTINGOWNER,1323957,14,24
3,FOOTNOTES,2865764,4,24


table,FOOTNOTES,NONDERIV_TRANS,REPORTINGOWNER,SUBMISSION
quarter,,,,
2020q1,159716,105852,73693,69408
2020q2,113675,70429,55817,51459
2020q3,92931,60619,43766,39096
2020q4,108077,72973,49045,43607
2021q1,188895,125620,81046,74944
2021q2,140976,89108,66036,58798
2021q3,111270,74002,50774,44057
2021q4,118799,80096,52858,46675
2022q1,171741,110327,76260,71395


Transaction columns: ['ACCESSION_NUMBER', 'NONDERIV_TRANS_SK', 'SECURITY_TITLE', 'SECURITY_TITLE_FN', 'TRANS_DATE', 'TRANS_DATE_FN', 'DEEMED_EXECUTION_DATE', 'DEEMED_EXECUTION_DATE_FN', 'TRANS_FORM_TYPE', 'TRANS_CODE', 'EQUITY_SWAP_INVOLVED', 'EQUITY_SWAP_TRANS_CD_FN', 'TRANS_TIMELINESS', 'TRANS_TIMELINESS_FN', 'TRANS_SHARES', 'TRANS_SHARES_FN', 'TRANS_PRICEPERSHARE', 'TRANS_PRICEPERSHARE_FN', 'TRANS_ACQUIRED_DISP_CD', 'TRANS_ACQUIRED_DISP_CD_FN', 'SHRS_OWND_FOLWNG_TRANS', 'SHRS_OWND_FOLWNG_TRANS_FN', 'VALU_OWND_FOLWNG_TRANS', 'VALU_OWND_FOLWNG_TRANS_FN', 'DIRECT_INDIRECT_OWNERSHIP', 'DIRECT_INDIRECT_OWNERSHIP_FN', 'NATURE_OF_OWNERSHIP', 'NATURE_OF_OWNERSHIP_FN', 'SOURCE_QUARTER']


,ACCESSION_NUMBER,NONDERIV_TRANS_SK,SECURITY_TITLE,SECURITY_TITLE_FN,TRANS_DATE,TRANS_DATE_FN,DEEMED_EXECUTION_DATE,DEEMED_EXECUTION_DATE_FN,TRANS_FORM_TYPE,TRANS_CODE,...,TRANS_ACQUIRED_DISP_CD_FN,SHRS_OWND_FOLWNG_TRANS,SHRS_OWND_FOLWNG_TRANS_FN,VALU_OWND_FOLWNG_TRANS,VALU_OWND_FOLWNG_TRANS_FN,DIRECT_INDIRECT_OWNERSHIP,DIRECT_INDIRECT_OWNERSHIP_FN,NATURE_OF_OWNERSHIP,NATURE_OF_OWNERSHIP_FN,SOURCE_QUARTER
0,0001446472-20-000002,281943,Common Stock,<NA>,31-MAR-2020,<NA>,<NA>,<NA>,4,P,...,<NA>,1537800.0,F2,<NA>,<NA>,D,<NA>,<NA>,<NA>,2020q1
1,0001084991-20-000014,379212,Common Stock,<NA>,31-MAR-2020,<NA>,<NA>,<NA>,4,M,...,<NA>,8560.0,<NA>,<NA>,<NA>,I,<NA>,By Rabbi Trust,<NA>,2020q1
2,0001104659-20-041411,245379,Common Stock,<NA>,28-MAR-2020,<NA>,<NA>,<NA>,4,F,...,<NA>,54890.0,<NA>,<NA>,<NA>,D,<NA>,<NA>,<NA>,2020q1
3,0001104659-20-041411,245380,Common Stock,<NA>,31-MAR-2020,<NA>,<NA>,<NA>,4,F,...,<NA>,54338.0,<NA>,<NA>,<NA>,D,<NA>,<NA>,<NA>,2020q1
4,0001104659-20-041409,272457,Common Stock,<NA>,28-MAR-2020,<NA>,<NA>,<NA>,4,F,...,<NA>,24497.0,<NA>,<NA>,<NA>,D,<NA>,<NA>,<NA>,2020q1


### Preview the extracted SEC tables

Show the first five rows of each extracted SEC DataFrame.


In [4]:
display(df_submission.head())
display(df_transactions.head())
display(df_owners.head())
display(df_footnotes.head())


,ACCESSION_NUMBER,FILING_DATE,PERIOD_OF_REPORT,DATE_OF_ORIG_SUB,NO_SECURITIES_OWNED,NOT_SUBJECT_SEC16,FORM3_HOLDINGS_REPORTED,FORM4_TRANS_REPORTED,DOCUMENT_TYPE,ISSUERCIK,ISSUERNAME,ISSUERTRADINGSYMBOL,REMARKS,SOURCE_QUARTER,AFF10B5ONE
0,0001047340-20-000142,31-MAR-2020,27-MAR-2020,<NA>,<NA>,0,<NA>,<NA>,4,0001047340,FRESH DEL MONTE PRODUCE INC,FDP,<NA>,2020q1,<NA>
1,0001209191-20-021789,31-MAR-2020,27-MAR-2020,<NA>,<NA>,0,<NA>,<NA>,4,0001400810,"HCI Group, Inc.",HCI,<NA>,2020q1,<NA>
2,0001593034-20-000030,31-MAR-2020,29-MAR-2020,<NA>,<NA>,0,<NA>,<NA>,4,0001593034,Endo International plc,ENDP,<NA>,2020q1,<NA>
3,0001726978-20-000119,31-MAR-2020,27-MAR-2020,<NA>,<NA>,0,<NA>,<NA>,4,0001726978,"Goosehead Insurance, Inc.",GSHD,The sales reported were effected pursuant to a...,2020q1,<NA>
4,0001289419-20-000061,31-MAR-2020,27-MAR-2020,<NA>,<NA>,<NA>,<NA>,<NA>,4,0001289419,"Morningstar, Inc.",MORN,<NA>,2020q1,<NA>


,ACCESSION_NUMBER,NONDERIV_TRANS_SK,SECURITY_TITLE,SECURITY_TITLE_FN,TRANS_DATE,TRANS_DATE_FN,DEEMED_EXECUTION_DATE,DEEMED_EXECUTION_DATE_FN,TRANS_FORM_TYPE,TRANS_CODE,...,TRANS_ACQUIRED_DISP_CD_FN,SHRS_OWND_FOLWNG_TRANS,SHRS_OWND_FOLWNG_TRANS_FN,VALU_OWND_FOLWNG_TRANS,VALU_OWND_FOLWNG_TRANS_FN,DIRECT_INDIRECT_OWNERSHIP,DIRECT_INDIRECT_OWNERSHIP_FN,NATURE_OF_OWNERSHIP,NATURE_OF_OWNERSHIP_FN,SOURCE_QUARTER
0,0001446472-20-000002,281943,Common Stock,<NA>,31-MAR-2020,<NA>,<NA>,<NA>,4,P,...,<NA>,1537800.0,F2,<NA>,<NA>,D,<NA>,<NA>,<NA>,2020q1
1,0001084991-20-000014,379212,Common Stock,<NA>,31-MAR-2020,<NA>,<NA>,<NA>,4,M,...,<NA>,8560.0,<NA>,<NA>,<NA>,I,<NA>,By Rabbi Trust,<NA>,2020q1
2,0001104659-20-041411,245379,Common Stock,<NA>,28-MAR-2020,<NA>,<NA>,<NA>,4,F,...,<NA>,54890.0,<NA>,<NA>,<NA>,D,<NA>,<NA>,<NA>,2020q1
3,0001104659-20-041411,245380,Common Stock,<NA>,31-MAR-2020,<NA>,<NA>,<NA>,4,F,...,<NA>,54338.0,<NA>,<NA>,<NA>,D,<NA>,<NA>,<NA>,2020q1
4,0001104659-20-041409,272457,Common Stock,<NA>,28-MAR-2020,<NA>,<NA>,<NA>,4,F,...,<NA>,24497.0,<NA>,<NA>,<NA>,D,<NA>,<NA>,<NA>,2020q1


,ACCESSION_NUMBER,RPTOWNERCIK,RPTOWNERNAME,RPTOWNER_RELATIONSHIP,RPTOWNER_TITLE,RPTOWNER_TXT,RPTOWNER_STREET1,RPTOWNER_STREET2,RPTOWNER_CITY,RPTOWNER_STATE,RPTOWNER_ZIPCODE,RPTOWNER_STATE_DESC,FILE_NUMBER,SOURCE_QUARTER
0,0001446472-20-000002,0001441252,Pratt Brian,TenPercentOwner,<NA>,<NA>,"5950 BERKSHIRE LANE, STE. 800",<NA>,DALLAS,TX,75225,<NA>,001-36541,2020q1
1,0001084991-20-000014,0001271773,HUGHES WILLIAM F JR,Director,<NA>,<NA>,42921 NORMANDY LANE,<NA>,LANCASTER,CA,93536,<NA>,001-31398,2020q1
2,0001104659-20-041411,0001579464,Schrepfer Robert W,Officer,SVP - New Bus Dev & Spec Sales,<NA>,"C/O ANI PHARMACEUTICALS, INC.",210 MAIN STREET WEST,BAUDETTE,MN,56623,<NA>,001-31812,2020q1
3,0001104659-20-041409,0001674236,CAREY STEPHEN P.,Officer,Vice President and CFO,<NA>,"C/O ANI PHARMACEUTICALS, INC",210 MAIN STREET WEST,BAUDETTE,MN,56623,<NA>,001-31812,2020q1
4,0001209191-20-021971,0001055565,STULL STEVEN T,Director,<NA>,<NA>,3280 PEACHTREE RD.,SUITE 2400,ATLANTA,GA,30305,<NA>,001-35004,2020q1


,ACCESSION_NUMBER,FOOTNOTE_ID,FOOTNOTE_TXT,SOURCE_QUARTER
0,0001446472-20-000002,F1,The price reported in Column 4 is a weighted a...,2020q1
1,0001446472-20-000002,F2,These shares are held as community property by...,2020q1
2,0001084991-20-000014,F2,Not Applicable. Each RSU represents the right ...,2020q1
3,0001084991-20-000014,F1,The shares were issued in connection with the ...,2020q1
4,0001104659-20-041411,F2,Shares withheld for tax purposes exempt under ...,2020q1


### Loading historical S&P 500 membership

The CSV in `data/raw/historical/` contains a `date` and a comma-separated list of
`tickers` for each membership snapshot. Read it with `pd.read_csv`; Pandas preserves
that quoted ticker list as one column.

These are dated snapshots, not a row for every trading day. When matching filings
later, use the most recent snapshot on or before the disclosure date, and handle
historical ticker changes explicitly. We are only loading and inspecting the file here.

Source: [fja05680/sp500 historical membership](https://github.com/fja05680/sp500).

In [5]:
# Load and inspect the supplied historical membership CSV.
from pathlib import Path

raw_dir = Path("data/raw")
if not raw_dir.is_dir():
    raw_dir = Path("insider-trading-phase2/data/raw")

historical_file = raw_dir / "historical" / "S&P 500 Historical Components & Changes (Updated).csv"
df_historical = pd.read_csv(historical_file)
historical_dates = pd.to_datetime(df_historical["date"], format="%Y-%m-%d")
print(f"Historical membership: {len(df_historical):,} rows, {df_historical.shape[1]} columns")
print(f"Snapshot dates: {historical_dates.min().date()} to {historical_dates.max().date()}")
print("Missing values:", df_historical.isna().sum().to_dict())
display(df_historical.head())

Historical membership: 2,720 rows, 2 columns
Snapshot dates: 1996-01-02 to 2026-08-18
Missing values: {'date': 0, 'tickers': 0}


,date,tickers
0,1996-01-02,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
1,1996-01-03,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
2,1996-01-04,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
3,1996-01-10,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
4,1996-01-11,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."


# __DATA COLLECTION AND CLEANING APPENDIX__

### Collection process

[Describe which SEC quarters and stock-price histories were obtained, how they
were downloaded, and when. Document the company-selection rule and the raw filenames.
Describe collection in text; include the cleaning code below.]

### Cleaning historical S&P 500 membership

We clean the membership data separately from the SEC transactions. The original
`df_historical` and source CSV are preserved.

1. Parse the snapshot dates and normalize ticker whitespace/capitalization.
2. Remove repeated tickers within a snapshot and exact duplicate snapshots.
   Stop for missing values, empty membership lists, or conflicting snapshots on
   the same date, because silently dropping them could create incorrect coverage.
3. Derive the study boundaries from `FILING_DATE` in the loaded SEC batches
   (`2020q1` through `2025q4`), rather than treating a trading calendar as the
   source of the quarter definitions.
4. Keep the latest snapshot on or before the first filing date, plus all subsequent
   snapshots through the last filing date. The starting snapshot can be dated
   before 2020 because its membership remains in effect until the next change.

We initially treat the community dataset's snapshot date as the date on which its
membership takes effect. Confirm that convention against a sample of original
index announcements before using it for inference. Ticker punctuation is preserved;
this step does not resolve ticker renames or map symbols to a price provider.

In [6]:
# Clean historical membership snapshots and retain coverage for the actual SEC filing window.
filing_dates = pd.to_datetime(df_submission["FILING_DATE"], format="mixed", errors="raise")
if filing_dates.isna().any() or filing_dates.empty:
    raise ValueError("Review missing SEC filing dates before defining the study window.")
study_start = filing_dates.min().normalize()
study_end = filing_dates.max().normalize()
study_end_exclusive = study_end + pd.Timedelta(days=1)

historical = df_historical[["date", "tickers"]].copy()
if historical.isna().any().any():
    raise ValueError("Review missing snapshot dates or ticker lists before cleaning.")
historical["date"] = pd.to_datetime(historical["date"], format="%Y-%m-%d", errors="raise")
ticker_lists = historical["tickers"].astype("string").str.split(",")
ticker_lists = ticker_lists.map(lambda values: [ticker.strip().upper() for ticker in values if ticker.strip()])
repeated_tickers = int(ticker_lists.map(lambda values: len(values) - len(set(values))).sum())
if ticker_lists.map(len).eq(0).any():
    raise ValueError("An empty membership list requires review.")
historical["tickers"] = ticker_lists.map(lambda values: ",".join(sorted(set(values))))
exact_duplicates = int(historical.duplicated(["date", "tickers"]).sum())
historical = historical.drop_duplicates(["date", "tickers"]).sort_values("date")
if historical["date"].duplicated().any():
    raise ValueError("Conflicting membership snapshots share the same date; review them first.")

baseline = historical.loc[historical["date"].le(study_start)].tail(1)
if baseline.empty:
    raise ValueError("A membership snapshot on or before the first filing is required.")
changes = historical.loc[historical["date"].gt(study_start) & historical["date"].le(study_end)]
historical_snapshots = pd.concat([baseline, changes], ignore_index=True)
historical_snapshots = historical_snapshots.rename(columns={"date": "snapshot_date"})
historical_snapshots["valid_from"] = historical_snapshots["snapshot_date"].clip(lower=study_start)
historical_snapshots["valid_to"] = historical_snapshots["snapshot_date"].shift(-1).fillna(study_end_exclusive)

print(f"SEC filing window: {study_start.date()} through {study_end.date()}")
print(f"Starting membership snapshot: {baseline['date'].iloc[0].date()}")
print(f"Original snapshots: {len(df_historical):,}; retained snapshots: {len(historical_snapshots):,}")
print(f"Exact duplicate snapshots removed: {exact_duplicates}; repeated ticker entries removed: {repeated_tickers}")
display(historical_snapshots[["snapshot_date", "valid_from", "valid_to"]].head())

SEC filing window: 2020-01-02 through 2025-12-31
Starting membership snapshot: 2019-12-23
Original snapshots: 2,720; retained snapshots: 91
Exact duplicate snapshots removed: 0; repeated ticker entries removed: 0


,snapshot_date,valid_from,valid_to
0,2019-12-23,2020-01-02,2020-01-28
1,2020-01-28,2020-01-28,2020-03-03
2,2020-03-03,2020-03-03,2020-04-03
3,2020-04-03,2020-04-03,2020-04-06
4,2020-04-06,2020-04-06,2020-05-12


### One ticker per membership interval

We split each retained snapshot into one row per ticker. The cleaned table
`df_historical_clean` contains:

- **`ticker`:** Symbol recorded in the historical membership source.
- **`snapshot_date`:** Date of the source snapshot.
- **`valid_from`:** First included date in this study window.
- **`valid_to`:** First date no longer covered by this snapshot; **exclusive**.

A disclosure matches a row when `valid_from <= FILING_DATE < valid_to` and the
company symbol has been mapped correctly. For example, `valid_to = 2026-01-01`
includes December 31, 2025. These are **snapshot coverage intervals**, not claimed
original company admission or exit dates. A company remaining in the index across
several snapshots will correctly have several adjacent rows.
The code below removes rows with missing values and duplicate ticker/start-date
pairs, clips intervals to the study window, and shortens any overlapping interval
to end at the next start date for that ticker. It then removes empty intervals.


In [7]:
# Expand the retained snapshots into a tidy ticker-by-interval table.
df_historical_clean = historical_snapshots.assign(
    ticker=historical_snapshots["tickers"].str.split(",")
).explode("ticker", ignore_index=True)
df_historical_clean = df_historical_clean[
    ["ticker", "snapshot_date", "valid_from", "valid_to"]
]

# Remove incomplete rows and repeated ticker/start-date pairs.
df_historical_clean = (
    df_historical_clean.dropna()
    .sort_values(["ticker", "valid_from"])
    .drop_duplicates(["ticker", "valid_from"])
    .reset_index(drop=True)
)

# Keep intervals within the study window and end them by the next interval.
df_historical_clean["valid_from"] = df_historical_clean["valid_from"].clip(lower=study_start)
df_historical_clean["valid_to"] = df_historical_clean["valid_to"].clip(upper=study_end_exclusive)
next_start = df_historical_clean.groupby("ticker")["valid_from"].shift(-1)
df_historical_clean["valid_to"] = df_historical_clean["valid_to"].where(
    next_start.isna() | df_historical_clean["valid_to"].le(next_start), next_start
)

# Remove intervals that have no days remaining.
df_historical_clean = df_historical_clean.loc[
    df_historical_clean["valid_from"].lt(df_historical_clean["valid_to"])
].reset_index(drop=True)

print(f"Cleaned membership rows: {len(df_historical_clean):,}")
print(f"Distinct ticker symbols during the study window: {df_historical_clean['ticker'].nunique():,}")
print("Ticker counts are not necessarily unique company counts: symbols and share classes can change.")
display(df_historical_clean.head(10))


Cleaned membership rows: 45,850
Distinct ticker symbols during the study window: 627
Ticker counts are not necessarily unique company counts: symbols and share classes can change.


,ticker,snapshot_date,valid_from,valid_to
0,A,2019-12-23,2020-01-02,2020-01-28
1,A,2020-01-28,2020-01-28,2020-03-03
2,A,2020-03-03,2020-03-03,2020-04-03
3,A,2020-04-03,2020-04-03,2020-04-06
4,A,2020-04-06,2020-04-06,2020-05-12
5,A,2020-05-12,2020-05-12,2020-05-22
6,A,2020-05-22,2020-05-22,2020-06-22
7,A,2020-06-22,2020-06-22,2020-09-18
8,A,2020-09-18,2020-09-18,2020-09-21
9,A,2020-09-21,2020-09-21,2020-10-07


### Group by ticker

Combine consecutive snapshots for the same ticker into one continuous membership
period. Most tickers then have one row. If a ticker disappears and later appears
again, keep separate rows so a later join does not fill the gap incorrectly.

`valid_from` and `valid_to` now cover the combined period, with `valid_to` still
exclusive. `snapshot_date` is the earliest source snapshot in that period.
Running the cell below also saves the grouped table to
`data/processed/historical_membership_clean.csv`, replacing the previous version.


In [8]:
# Combine consecutive snapshots into continuous membership periods by ticker.
snapshot_rows = len(df_historical_clean)
previous_end = df_historical_clean.groupby("ticker")["valid_to"].shift()
df_historical_clean["membership_period"] = (
    df_historical_clean["valid_from"].ne(previous_end)
    .groupby(df_historical_clean["ticker"]).cumsum()
)
df_historical_clean = (
    df_historical_clean.groupby(["ticker", "membership_period"], as_index=False)
    .agg(snapshot_date=("snapshot_date", "min"),
         valid_from=("valid_from", "min"),
         valid_to=("valid_to", "max"))
    .drop(columns="membership_period")
)

print(f"Rows before grouping: {snapshot_rows:,}; membership periods: {len(df_historical_clean):,}")
print(f"Distinct ticker symbols: {df_historical_clean['ticker'].nunique():,}")
display(df_historical_clean.head(10))

# Save the cleaned historical membership intervals for reuse.
processed_dir = raw_dir.parent / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
df_historical_clean.to_csv(processed_dir / "historical_membership_clean.csv", index=False)
print("Saved data/processed/historical_membership_clean.csv")

Rows before grouping: 45,850; membership periods: 628
Distinct ticker symbols: 627


,ticker,snapshot_date,valid_from,valid_to
0,A,2019-12-23,2020-01-02,2026-01-01
1,AAL,2019-12-23,2020-01-02,2024-09-23
2,AAP,2019-12-23,2020-01-02,2023-08-25
3,AAPL,2019-12-23,2020-01-02,2026-01-01
4,ABBV,2019-12-23,2020-01-02,2026-01-01
5,ABC,2019-12-23,2020-01-02,2023-08-30
6,ABMD,2019-12-23,2020-01-02,2022-12-22
7,ABNB,2023-09-18,2023-09-18,2026-01-01
8,ABT,2019-12-23,2020-01-02,2026-01-01
9,ACGL,2022-11-01,2022-11-01,2026-01-01


Saved data/processed/historical_membership_clean.csv


### Collecting stock prices (described here, run separately)

The rubric asks for collection to be **described** rather than shown as code, so the download lives in `collect_prices.py` in this folder and the notebook only reads its saved output.

**What the script does:**

- **Which tickers:** every ticker that was in the S&P 500 at any point in our study window (627 symbols from `data/processed/historical_membership_clean.csv`), plus **SPY**, an exchange-traded fund that tracks the S&P 500. SPY is our measure of "the market".
- **Source and frequency:** *daily* prices from Yahoo Finance through the `yfinance` library, from **January 1, 2019** onward. The extra year before our first filing is needed to estimate each stock's normal relationship with the market (Step 4).
- **Adjustments:** `auto_adjust=True` returns prices adjusted for stock splits and dividends, so a return computed from them is what a shareholder actually earned. Without this, a 2-for-1 split would look like a 50% crash.
- **Missing symbols:** symbols that fail in a batch are retried one at a time. Any symbol that still returns nothing, or fewer than 20 trading days (a Yahoo glitch that on our download date returned a single day for AvalonBay, EA, and a few others), is treated as missing and listed in `daily_stock_prices_missing.csv`. Most missing symbols are companies acquired, taken private, or renamed before our download date (for example Twitter, Activision Blizzard, SVB Financial).
- **When:** downloaded once on October 19, 2026, and saved to `data/raw/prices/daily_stock_prices.csv.gz` (one row per ticker per trading day).

*This replaces our first weekly download (`weekly_stock_prices.csv`, kept in the folder for reference). Yahoo's weekly bars were anchored to whatever weekday the download started on, so we now build standard Friday-to-Friday weeks ourselves from daily data.*

In [9]:
# Load the saved daily prices and the list of symbols Yahoo could not return.
df_prices_daily = pd.read_csv(raw_dir / "prices" / "daily_stock_prices.csv.gz", parse_dates=["date"])
missing_price_tickers = pd.read_csv(raw_dir / "prices" / "daily_stock_prices_missing.csv")["ticker"].tolist()
print(f"Daily price rows: {len(df_prices_daily):,}; tickers with data: {df_prices_daily['ticker'].nunique()} (including SPY)")
print(f"Date range: {df_prices_daily['date'].min().date()} to {df_prices_daily['date'].max().date()}")
print(f"Symbols with no Yahoo data ({len(missing_price_tickers)}):", missing_price_tickers)
display(df_prices_daily.head())

Daily price rows: 1,058,363; tickers with data: 553 (including SPY)
Date range: 2019-01-02 to 2026-10-19
Symbols with no Yahoo data (75): ['ABC', 'ABMD', 'ADS', 'AGN', 'ALXN', 'ANSS', 'ANTM', 'ARNC', 'ATVI', 'AVB', 'BK', 'BLL', 'CDAY', 'CERN', 'CMA', 'COG', 'CTL', 'CTLT', 'CTRA', 'CTXS', 'CXO', 'DAY', 'DFS', 'DISCA', 'DISCK', 'DISH', 'DRE', 'EA', 'EQR', 'ETFC', 'FBHS', 'FI', 'FLIR', 'FLT', 'FRC', 'GPS', 'HBI', 'HES', 'HFC', 'HOLX', 'IPG', 'JNPR', 'JWN', 'K', 'KSU', 'LEG', 'MMC', 'MRO', 'MXIM', 'MYL', 'NBL', 'NLOK', 'NLSN', 'PBCT', 'PEAK', 'PKI', 'PSKY', 'PXD', 'QRVO', 'RE', 'RTN', 'SEE', 'SIVB', 'TIF', 'TWTR', 'UTX', 'VAR', 'VIAC', 'WBA', 'WBD', 'WCG', 'WLTW', 'WRK', 'XEC', 'XLNX']


,ticker,date,close,volume
0,A,2019-01-02,62.087963,2113300.0
1,A,2019-01-03,59.800667,5383900.0
2,A,2019-01-04,61.870571,3123700.0
3,A,2019-01-07,63.184349,3235100.0
4,A,2019-01-08,64.110641,1578100.0


### Step 1: Cleaning the SEC transactions

The raw `NONDERIV_TRANS` table holds **every** non-derivative transaction reported on Forms 3, 4, and 5: grants, tax withholdings, gifts, option exercises, and real trades. Our research question is about **open-market buying and selling that the public learns about through a Form 4**, so this step narrows 1.84 million raw rows down to clean, analysis-eligible trades.

Each sub-step below has one job, prints how many rows it keeps, and states *why* the rows it drops cannot answer our question. The raw DataFrames loaded above are never modified; every step works on a copy.

#### 1a. Attach filing information and keep Form 4 filings only

A transaction row on its own does not say **which company** it belongs to or **when the public saw it**. Those fields live in `SUBMISSION`, one row per filing, linked by `ACCESSION_NUMBER` (the SEC's unique ID for one filing). We bring over:

- `FILING_DATE`: when the filing reached EDGAR. This is our event date, because it is the first moment an outside investor could act.
- `ISSUERCIK` and `ISSUERTRADINGSYMBOL`: the company's permanent SEC ID (CIK) and its ticker, used later to match S&P 500 membership and prices. Some quarters store the CIK without leading zeros (`38777` instead of `0000038777`), so we pad every CIK to 10 digits; otherwise one company would look like two.
- `AFF10B5ONE`: a checkbox the SEC added in April 2023 marking trades made under a pre-scheduled Rule 10b5-1 plan.

We then keep only `DOCUMENT_TYPE` **4** and **4/A** (an amended Form 4). **Form 3** is an insider's initial holdings statement and reports no trades. **Form 5** is an annual catch-all filed up to 45 days after fiscal year-end, so its "disclosure date" can lag the trade by over a year, and it accounts for under 1% of filings.

In [10]:
# 1a. Merge filing-level fields onto each transaction, then keep Forms 4 and 4/A.
submission_cols = ["ACCESSION_NUMBER", "FILING_DATE", "DOCUMENT_TYPE", "ISSUERCIK",
                   "ISSUERNAME", "ISSUERTRADINGSYMBOL", "AFF10B5ONE"]
trades = df_transactions.merge(df_submission[submission_cols], on="ACCESSION_NUMBER",
                               how="left", validate="many_to_one")
assert trades["FILING_DATE"].notna().all(), "Every transaction should match exactly one filing."
# Some quarters store CIKs without leading zeros (38777 vs 0000038777); pad so one company has one ID.
trades["ISSUERCIK"] = trades["ISSUERCIK"].str.strip().str.zfill(10)

cleaning_log = [{"step": "Raw non-derivative transactions", "rows": len(trades)}]
trades = trades[trades["DOCUMENT_TYPE"].isin(["4", "4/A"])].copy()
cleaning_log.append({"step": "1a. Forms 4 and 4/A only", "rows": len(trades)})
print(trades["DOCUMENT_TYPE"].value_counts().to_string())

DOCUMENT_TYPE
4      1793464
4/A      26865


#### 1b. Keep open-market purchases (P) and sales (S)

`TRANS_CODE` records *why* shares changed hands. Only two codes reflect an insider **choosing** to put their own money in or take it out at the market price:

| Code | Meaning | Kept? | Why |
|---|---|---|---|
| **P** | Open-market or private purchase | Yes | The insider paid cash at the market price. |
| **S** | Open-market or private sale | Yes | The insider chose to sell at the market price. |
| A | Grant or award from the company | No | Compensation, not a decision about the stock's value. |
| F | Shares withheld to pay taxes | No | Triggered automatically by payroll when stock vests. |
| M | Option exercise | No | Converts an option into shares; the decision was made at grant. |
| G | Gift | No | No price is paid, so it says nothing about value. |

Counting an `F` tax withholding as a "sale" would create a signal generated by **payroll** rather than by **conviction**, so these codes are excluded rather than mixed in.

In [11]:
# 1b. Keep only open-market purchases (P) and sales (S).
print("Transaction codes in Form 4 rows (top 8):")
print(trades["TRANS_CODE"].value_counts().head(8).to_string())
trades = trades[trades["TRANS_CODE"].isin(["P", "S"])].copy()
cleaning_log.append({"step": "1b. Purchase (P) and sale (S) codes only", "rows": len(trades)})

Transaction codes in Form 4 rows (top 8):
TRANS_CODE
S    546649
A    383521
F    296238
M    265436
P    164790
J     49083
G     38632
D     36197


#### 1c. Convert text fields to dates and numbers, and drop unusable rows

Everything was loaded as text to protect identifiers like CIKs with leading zeros. Now we convert:

- **Dates** (`TRANS_DATE`, `FILING_DATE`) from the SEC's `31-MAR-2020` format to real dates.
- **Numbers** (`TRANS_SHARES`, `TRANS_PRICEPERSHARE`, `SHRS_OWND_FOLWNG_TRANS`) to floats.

Then we compute two new columns:

- `trade_value` = shares × price per share, the dollar size of the trade.
- `filing_lag_days` = filing date − transaction date. The legal deadline is **two business days**, so this column lets us check timeliness.

We drop three kinds of unusable rows:

- **Shares or price missing or ≤ 0.** A trade with no price has no dollar size and is usually a reporting error.
- **Filed before the trade date.** This is impossible and signals a data-entry error.
- **Filed more than 365 days after the trade.** The legal deadline is two business days, and 99% of trades are filed within about 7 months. Gaps beyond a year are dominated by typos, such as a transaction year entered as `24` instead of `2024`, which produces a "lag" of 2,000 years.

Late filings inside that window are **kept**, because our event date is the filing date: a late filing simply means the public learned later.

In [12]:
# 1c. Parse dates and numbers, compute trade value and filing lag, drop unusable rows.
for col in ["TRANS_DATE", "FILING_DATE"]:
    trades[col] = pd.to_datetime(trades[col], format="%d-%b-%Y", errors="coerce")
for col in ["TRANS_SHARES", "TRANS_PRICEPERSHARE", "SHRS_OWND_FOLWNG_TRANS"]:
    trades[col] = pd.to_numeric(trades[col], errors="coerce")

trades["trade_value"] = trades["TRANS_SHARES"] * trades["TRANS_PRICEPERSHARE"]
trades["filing_lag_days"] = (trades["FILING_DATE"] - trades["TRANS_DATE"]).dt.days

valid = (trades["TRANS_DATE"].notna() & trades["FILING_DATE"].notna()
         & trades["TRANS_SHARES"].gt(0) & trades["TRANS_PRICEPERSHARE"].gt(0)
         & trades["filing_lag_days"].between(0, 365))
print(f"Rows dropped (bad shares/price, unparseable dates, or lag outside 0-365 days): {(~valid).sum():,}")
trades = trades[valid].copy()
cleaning_log.append({"step": "1c. Valid dates (lag 0-365 days), shares > 0, price > 0", "rows": len(trades)})
print(trades["filing_lag_days"].describe(percentiles=[.5, .9, .99]).round(1).to_string())

Rows dropped (bad shares/price, unparseable dates, or lag outside 0-365 days): 9,497


count    700020.0
mean          4.2
std          20.2
min           0.0
50%           2.0
90%           4.0
99%          78.0
max         365.0


#### 1d. Resolve amended filings (Form 4/A)

A **Form 4/A** corrects an earlier Form 4. Because both versions stay on EDGAR, stacking quarters would count the same trade twice, once as originally filed and once as corrected.

**Rule:** for each *(company, insider, transaction date, transaction code)*, if any filing in that group is an amendment, we keep only the rows from the **most recently filed** version and drop the rest. Groups with no amendment are left untouched, because an insider often makes several genuine same-day trades at different prices, and collapsing those would delete real data.

The insider is identified by `RPTOWNERCIK`, their permanent SEC ID, from `REPORTINGOWNER`. When one filing lists several owners (for example, a fund and its manager), we use the lowest CIK so the group key is stable. Roles for **all** owners are attached in Step 2.

In [13]:
# 1d. Drop superseded rows when a Form 4/A amends an earlier filing.
first_owner = (df_owners.sort_values("RPTOWNERCIK")
               .drop_duplicates("ACCESSION_NUMBER")[["ACCESSION_NUMBER", "RPTOWNERCIK"]])
trades = trades.merge(first_owner, on="ACCESSION_NUMBER", how="left", validate="many_to_one")
trades["RPTOWNERCIK"] = trades["RPTOWNERCIK"].str.strip().str.zfill(10)

group_key = ["ISSUERCIK", "RPTOWNERCIK", "TRANS_DATE", "TRANS_CODE"]
trades["is_amendment"] = trades["DOCUMENT_TYPE"].eq("4/A")
group_has_amendment = trades.groupby(group_key, dropna=False)["is_amendment"].transform("any")

# Latest filing in each group; ties broken by the larger accession number.
latest_accession = (trades.sort_values(["FILING_DATE", "ACCESSION_NUMBER"])
                    .groupby(group_key, dropna=False)["ACCESSION_NUMBER"].transform("last"))
superseded = group_has_amendment & trades["ACCESSION_NUMBER"].ne(latest_accession)
print(f"Groups containing an amendment: {trades.loc[group_has_amendment, group_key].drop_duplicates().shape[0]:,}")
print(f"Superseded rows dropped: {superseded.sum():,}")
trades = trades[~superseded].copy()
cleaning_log.append({"step": "1d. Superseded amendment rows removed", "rows": len(trades)})

Groups containing an amendment: 5,706
Superseded rows dropped: 7,995


#### 1e. Tidy columns and save an intermediate file

We keep only the columns later steps need, give them readable lowercase names, and save the result to `data/interim/sec_ps_trades.csv.gz`. Saving here means Step 2 (attaching roles and S&P 500 membership) can restart from this file instead of re-reading 1.3 GB of raw TSVs.

`plan_10b5_1` is filled only for filings that carry the post-April-2023 checkbox; earlier filings are left missing rather than assumed "no plan." The cleaning log at the end shows every row removed along the way, so a reader can trace the full funnel from raw to clean.

In [14]:
# 1e. Select and rename columns, then save the cleaned trades.
df_trades_clean = trades.rename(columns={
    "ACCESSION_NUMBER": "accession", "ISSUERCIK": "issuer_cik", "ISSUERNAME": "issuer_name",
    "ISSUERTRADINGSYMBOL": "ticker_filed", "RPTOWNERCIK": "owner_cik",
    "TRANS_DATE": "trans_date", "FILING_DATE": "filing_date", "TRANS_CODE": "trans_code",
    "TRANS_SHARES": "shares", "TRANS_PRICEPERSHARE": "price", "SHRS_OWND_FOLWNG_TRANS": "shares_after",
    "SECURITY_TITLE": "security_title", "DOCUMENT_TYPE": "form_type", "SOURCE_QUARTER": "source_quarter",
})[["accession", "issuer_cik", "issuer_name", "ticker_filed", "owner_cik", "trans_date",
    "filing_date", "filing_lag_days", "trans_code", "shares", "price", "trade_value",
    "shares_after", "security_title", "form_type", "AFF10B5ONE", "source_quarter"]]

# Ticker cleanup: trim spaces, uppercase, and treat blanks or "NONE" as missing.
df_trades_clean["ticker_filed"] = (df_trades_clean["ticker_filed"].str.strip().str.upper()
                                   .replace({"": pd.NA, "NONE": pd.NA, "N/A": pd.NA}))
df_trades_clean["plan_10b5_1"] = pd.to_numeric(df_trades_clean.pop("AFF10B5ONE"), errors="coerce")
df_trades_clean = df_trades_clean.sort_values(["filing_date", "issuer_cik", "accession"]).reset_index(drop=True)

interim_dir = raw_dir.parent / "interim"
interim_dir.mkdir(parents=True, exist_ok=True)
df_trades_clean.to_csv(interim_dir / "sec_ps_trades.csv.gz", index=False)

df_cleaning_log = pd.DataFrame(cleaning_log)
df_cleaning_log["rows_removed"] = -df_cleaning_log["rows"].diff().fillna(0).astype(int)
display(df_cleaning_log)
print(df_trades_clean["trans_code"].value_counts().rename({"P": "purchases", "S": "sales"}).to_string())
print(f"Saved {len(df_trades_clean):,} rows to data/interim/sec_ps_trades.csv.gz")
display(df_trades_clean.head())

,step,rows,rows_removed
0,Raw non-derivative transactions,1841674,0
1,1a. Forms 4 and 4/A only,1820329,21345
2,1b. Purchase (P) and sale (S) codes only,711439,1108890
3,"1c. Valid dates (lag 0-365 days), shares > 0, ...",700020,11419
4,1d. Superseded amendment rows removed,692025,7995


trans_code
sales        537365
purchases    154660
Saved 692,025 rows to data/interim/sec_ps_trades.csv.gz


,accession,issuer_cik,issuer_name,ticker_filed,owner_cik,trans_date,filing_date,filing_lag_days,trans_code,shares,price,trade_value,shares_after,security_title,form_type,source_quarter,plan_10b5_1
0,0001127602-20-000197,0000001750,AAR CORP,AIR,0001027741,2019-12-24,2020-01-02,9,S,4793.0,48.02,230159.86,547399.0,Common Stock,4,2020q1,<NA>
1,0000914190-20-000002,0000007623,ARTS WAY MANUFACTURING CO INC,ARTW,0001022585,2019-12-30,2020-01-02,3,P,1151.0,1.75,2014.25,2056982.0,Common Stock,4,2020q1,<NA>
2,0000914190-20-000002,0000007623,ARTS WAY MANUFACTURING CO INC,ARTW,0001022585,2019-12-31,2020-01-02,2,P,51.0,1.75,89.25,2057033.0,Common Stock,4,2020q1,<NA>
3,0001271163-20-000001,0000009521,BANCROFT FUND LTD,BCV,0001185533,2019-12-30,2020-01-02,3,S,1500.0,26.02,39030.0,150500.0,Common Shares,4,2020q1,<NA>
4,0001271163-20-000001,0000009521,BANCROFT FUND LTD,BCV,0001185533,2019-12-31,2020-01-02,2,S,2500.0,26.21,65525.0,148000.0,Common Shares,4,2020q1,<NA>


### Step 2: Attaching insider roles and S&P 500 membership

Step 1 produced clean trades for **every** company that files with the SEC, from mega-caps to tiny funds. Step 2 adds the two facts our research question needs about each trade:

1. **Who traded?** The insider's role (CEO, director, large shareholder, and so on), from `REPORTINGOWNER`. Our second preregistered analysis compares roles, so this column must be reliable.
2. **Was the company in the S&P 500 on the filing date?** Our price data covers S&P 500 members only (the Phase 1 scope decision), so trades in other companies cannot be matched to returns.

This is the step that **merges our two meaningfully different sources**: SEC regulatory filings and the historical index-membership list.

#### 2a. Classify each insider's role

`REPORTINGOWNER` has one row per insider per filing. Two fields describe the role:

- `RPTOWNER_RELATIONSHIP`: SEC checkboxes, any combination of *Director*, *Officer*, *TenPercentOwner* (owns more than 10% of the stock), and *Other*.
- `RPTOWNER_TITLE`: free text such as "President & CEO" or "EVP, Chief Financial Officer".

We sort every filing into **one** role group, checking from most to least senior:

| Role group | Rule | Why it matters |
|---|---|---|
| **C-suite** | Officer whose title names CEO, CFO, COO, or President (but not *Vice* President) | The executives with the broadest view of the company's prospects |
| **Other officer** | Any other officer (General Counsel, CTO, SVP, and so on) | Inside information, but narrower |
| **Director** | Board member who is not an officer | Sees strategy, not day-to-day operations |
| **10% owner** | Large shareholder who is neither officer nor director | Often an investment fund; may trade for portfolio reasons |
| **Other** | None of the above | Rare |

If one filing lists several owners (for example, a fund and its general partner), the filing takes the **most senior** role among them. We also keep separate 0/1 flags (`is_director`, `is_officer`, `is_ten_pct_owner`), because a person can be both a director and an officer.

In [15]:
# 2a. Build one role record per filing from REPORTINGOWNER.
owners = df_owners[["ACCESSION_NUMBER", "RPTOWNER_RELATIONSHIP", "RPTOWNER_TITLE"]].copy()
relationship = owners["RPTOWNER_RELATIONSHIP"].fillna("")
title = owners["RPTOWNER_TITLE"].fillna("").str.upper()

owners["is_director"] = relationship.str.contains("Director")
owners["is_officer"] = relationship.str.contains("Officer")
owners["is_ten_pct_owner"] = relationship.str.contains("TenPercentOwner")
# C-suite: CEO/CFO/COO by abbreviation or full name, or President that is not Vice President.
names_c_suite = title.str.contains(r"\b(?:CEO|CFO|COO)\b|CHIEF (?:EXECUTIVE|FINANCIAL|OPERATING) OFFICER")
is_president = title.str.contains(r"\bPRESIDENT\b") & ~title.str.contains(r"VICE[\s-]*PRES|\bVP\b")
owners["is_c_suite"] = owners["is_officer"] & (names_c_suite | is_president)

# Rank roles from most senior (1) to least (5); each filing keeps its most senior owner.
owners["role_rank"] = np.select(
    [owners["is_c_suite"], owners["is_officer"], owners["is_director"], owners["is_ten_pct_owner"]],
    [1, 2, 3, 4], default=5)
role_names = {1: "C-suite", 2: "Other officer", 3: "Director", 4: "10% owner", 5: "Other"}
df_roles = (owners.groupby("ACCESSION_NUMBER")
            .agg(role_rank=("role_rank", "min"), is_director=("is_director", "max"),
                 is_officer=("is_officer", "max"), is_ten_pct_owner=("is_ten_pct_owner", "max"),
                 n_owners=("role_rank", "size"), owner_title=("RPTOWNER_TITLE", "first"))
            .reset_index().rename(columns={"ACCESSION_NUMBER": "accession"}))
df_roles["role"] = df_roles["role_rank"].map(role_names)

trades_roles = df_trades_clean.merge(df_roles, on="accession", how="left", validate="many_to_one")
assert trades_roles["role"].notna().all(), "Every filing should have at least one reporting owner."
print(pd.crosstab(trades_roles["role"], trades_roles["trans_code"], margins=True).to_string())

trans_code          P       S     All
role                                 
10% owner       27225   48856   76081
C-suite         44168  196523  240691
Director        64139  115774  179913
Other            4338    4788    9126
Other officer   14790  171424  186214
All            154660  537365  692025


#### 2b. Match each trade to S&P 500 membership on its filing date

A trade belongs in our sample only if its company was **in the index on the day the filing became public**. Matching on today's index list would silently drop companies that later left, which are disproportionately the ones that did badly (*survivorship bias*).

Matching is harder than comparing tickers, because the two sources write tickers differently:

- **Share classes and punctuation:** the index lists `BRK.B`; Berkshire's filings say `BRKA`, `BRK.A`, or `BRKB`. Lennar files as `LEN, LEN.B`.
- **Spaced or bracketed tickers:** Northern Oil & Gas files as `N O G`; others write `(CALX)`.
- **Stray tickers:** a few unrelated small filers type a famous ticker by mistake (for example, a fund entering `A`, which is Agilent's ticker).

So we match in three passes:

1. **Normalize tickers.** Split each filed ticker field into its listed symbols, remove punctuation, and join spaced single letters (`N O G` becomes `NOG`). Index tickers are normalized the same way (`BRK.B` becomes `BRKB`).
2. **Link each index ticker to a company ID (CIK).** For each index ticker, count which companies used it in filings *while that ticker was in the index*. We keep every CIK that accounts for **at least 10%** of those filings. This drops strays (the fund typing `A` is 8% of `A` filings), but keeps genuine ID changes, such as BlackRock's new holding company in 2024 (15%) and old vs new Ingersoll Rand.
3. **Match trades by CIK and date.** A trade is in the sample when its company's CIK was linked to an index ticker whose membership interval covers the filing date (`valid_from <= filing_date < valid_to`).

Matching by CIK means all of a company's filings match, however the ticker was typed that day.

In [16]:
# 2b. Link index tickers to SEC company IDs (CIKs), then keep trades filed while the company was an S&P 500 member.
import re

def ticker_tokens(raw):
    """Split a filed ticker field into normalized symbols, e.g. 'LEN, LEN.B' -> ['LEN', 'LENB']."""
    if not isinstance(raw, str) or not raw.strip():
        return []
    text = raw.upper().strip()
    if re.fullmatch(r"[A-Z](?:\s[A-Z])+", text):   # 'N O G' -> 'NOG'
        text = text.replace(" ", "")
    parts = re.split(r"[\s,/_;]+|\bAND\b", text)
    return [re.sub(r"[^A-Z]", "", part) for part in parts if re.sub(r"[^A-Z]", "", part)]

# Pass 1: one row per (filing, normalized symbol); index tickers normalized the same way.
filings = trades_roles[["accession", "issuer_cik", "issuer_name", "ticker_filed", "filing_date"]].drop_duplicates("accession")
filings = filings.assign(symbol=filings["ticker_filed"].map(ticker_tokens)).explode("symbol").dropna(subset=["symbol"])
membership = df_historical_clean.assign(symbol=df_historical_clean["ticker"].str.replace(r"[^A-Z]", "", regex=True))

# Pass 2: count filings per (index ticker, CIK) while the ticker was in the index; keep CIKs with >= 10% share.
candidates = filings.merge(membership[["symbol", "ticker", "valid_from", "valid_to"]], on="symbol")
candidates = candidates[candidates["filing_date"].ge(candidates["valid_from"]) & candidates["filing_date"].lt(candidates["valid_to"])]
links = (candidates.groupby(["ticker", "issuer_cik"])
         .agg(n_filings=("accession", "nunique"), issuer_name=("issuer_name", "first")).reset_index())
links["share"] = links["n_filings"] / links.groupby("ticker")["n_filings"].transform("sum")
dropped_links = links[links["share"].lt(0.10)]
df_ticker_cik = links[links["share"].ge(0.10)]
print(f"Index tickers linked to a CIK: {df_ticker_cik['ticker'].nunique()} of {membership['ticker'].nunique()}")
print(f"Stray (ticker, CIK) links dropped for < 10% share: {len(dropped_links)}")
unlinked = sorted(set(membership["ticker"]) - set(df_ticker_cik["ticker"]))
print("Index tickers with no matching insider filings:", unlinked)

# Pass 3: attach membership intervals by CIK and keep trades whose filing date falls inside one.
cik_intervals = df_ticker_cik[["ticker", "issuer_cik"]].merge(membership[["ticker", "valid_from", "valid_to"]], on="ticker")
sp = trades_roles.merge(cik_intervals.rename(columns={"ticker": "sp_ticker"}), on="issuer_cik", how="inner")
sp = sp[sp["filing_date"].ge(sp["valid_from"]) & sp["filing_date"].lt(sp["valid_to"])]

# A company with two share classes in the index at once (GOOG and GOOGL) would appear twice; keep one row per trade,
# preferring the index ticker that matches what the insider filed.
sp["filed_symbol_match"] = [t in ticker_tokens(f) for t, f in zip(sp["sp_ticker"].str.replace(r"[^A-Z]", "", regex=True), sp["ticker_filed"])]
trade_key = ["accession", "issuer_cik", "owner_cik", "trans_date", "trans_code", "shares", "price", "security_title"]
sp = (sp.sort_values("filed_symbol_match", ascending=False)
        .drop_duplicates(trade_key).drop(columns=["valid_from", "valid_to", "filed_symbol_match"]))
cleaning_log.append({"step": "2b. Company in S&P 500 on filing date", "rows": len(sp)})
print(sp["trans_code"].value_counts().rename({"P": "purchases", "S": "sales"}).to_string())

Index tickers linked to a CIK: 611 of 627
Stray (ticker, CIK) links dropped for < 10% share: 8
Index tickers with no matching insider filings: ['AGN', 'AMTM', 'ARES', 'CPRI', 'CRH', 'CVNA', 'ERIE', 'FIX', 'FOXA', 'FRC', 'NWSA', 'PSKY', 'SBNY', 'SOLS', 'VNT', 'WCG']


trans_code
sales        121168
purchases      6361


#### 2c. Remove the same trade reported under two company IDs, and save

A few companies file insider forms under **two** related CIKs, for example PG&E Corp and its utility subsidiary Pacific Gas & Electric. The same person's trade can then appear twice with identical date, shares, and price. We keep one copy: rows are identical on *(insider, transaction date, code, shares, price, index ticker)*, so the duplicate adds no information.

The result, `data/interim/sp500_trades.csv.gz`, is every open-market S&P 500 insider trade from 2020 to 2025 with the insider's role attached. Step 4 rolls these trades up into events.

In [17]:
# 2c. Drop cross-CIK duplicates, then save the S&P 500 trade file.
dup_across_ciks = sp.duplicated(["owner_cik", "trans_date", "trans_code", "shares", "price", "sp_ticker"])
print(f"Duplicate trades reported under a second CIK: {dup_across_ciks.sum():,}")
df_sp_trades = sp[~dup_across_ciks].sort_values(["filing_date", "sp_ticker", "accession"]).reset_index(drop=True)
cleaning_log.append({"step": "2c. Same trade under two CIKs removed", "rows": len(df_sp_trades)})

df_sp_trades.to_csv(interim_dir / "sp500_trades.csv.gz", index=False)
df_ticker_cik.to_csv(interim_dir / "sp500_ticker_cik_links.csv", index=False)
print(f"Saved {len(df_sp_trades):,} trades in {df_sp_trades['issuer_cik'].nunique()} companies to data/interim/sp500_trades.csv.gz")
print(pd.crosstab(df_sp_trades["role"], df_sp_trades["trans_code"], margins=True).to_string())

Duplicate trades reported under a second CIK: 161


Saved 127,368 trades in 593 companies to data/interim/sp500_trades.csv.gz
trans_code        P       S     All
role                               
10% owner       742    3291    4033
C-suite        1205   51938   53143
Director       3633   23605   27238
Other            20     542     562
Other officer   758   41634   42392
All            6358  121010  127368


### Step 3: Cleaning prices and linking each company to one price series

Raw daily prices need three things before they can measure what happened after a filing:

1. **A quality check** for duplicate days and non-positive prices.
2. **Weekly returns.** A *return* is the percentage change in price: if a stock closes at $100 one Friday and $103 the next, its weekly return is 3%. We work in weeks rather than days because weekly returns are less noisy, and our horizons (1 to 26 weeks) are naturally counted in weeks.
3. **One price series per company.** Our trades are keyed by company ID (CIK), but prices are keyed by ticker, and tickers change.

#### 3a. Quality checks and Friday-to-Friday weekly returns

We drop exact duplicate ticker-days and any non-positive closing price, then keep each stock's **last closing price of every week ending Friday**. When a market holiday falls on a Friday, the week uses Thursday's close. The weekly return is that close divided by the previous week's close, minus 1.

The same is done for SPY, giving the **market return** for every week.

In [18]:
# 3a. Clean daily prices, then compute Friday-to-Friday weekly returns for every ticker and for SPY.
prices = df_prices_daily.drop_duplicates(["ticker", "date"])
bad_prices = prices["close"].le(0) | prices["close"].isna()
print(f"Duplicate ticker-days removed: {len(df_prices_daily) - len(prices):,}; non-positive or missing closes removed: {bad_prices.sum():,}")
prices = prices[~bad_prices]

# Last close of each week ending Friday ('W-FRI'), then the percentage change from the prior week.
weekly_close = (prices.set_index("date").groupby("ticker")["close"]
                .resample("W-FRI").last().dropna().reset_index())
weekly_close["ret"] = weekly_close.groupby("ticker")["close"].pct_change()
df_weekly = weekly_close.dropna(subset=["ret"]).rename(columns={"date": "week"})

market = df_weekly.loc[df_weekly["ticker"].eq("SPY"), ["week", "ret"]].rename(columns={"ret": "mkt_ret"})
df_weekly = df_weekly[df_weekly["ticker"].ne("SPY")].merge(market, on="week", how="inner")
print(f"Weekly return rows: {len(df_weekly):,} across {df_weekly['ticker'].nunique()} stocks; "
      f"{market['week'].min().date()} to {market['week'].max().date()}")
print("Weekly stock returns, percent:")
print((df_weekly["ret"] * 100).describe(percentiles=[.01, .5, .99]).round(2).to_string())

Duplicate ticker-days removed: 0; non-positive or missing closes removed: 0


Weekly return rows: 219,378 across 552 stocks; 2019-01-11 to 2026-10-23
Weekly stock returns, percent:
count    219378.00
mean          0.34
std           5.21
min         -61.01
1%          -13.87
50%           0.32
99%          15.20
max         129.20


#### 3b. Choose one price ticker for each company

A company can appear in the index under several tickers over time (Facebook `FB` became Meta `META`; AmerisourceBergen `ABC` became Cencora `COR`). Yahoo usually stores the **whole** price history under the **current** ticker and returns nothing for the old one.

Because Step 2 linked every index ticker to a company ID, we can fix this: for each company, among its linked index tickers that **have** price data, we use the one whose index membership ends **latest**, which is its most recent name. Companies with no ticker that has price data are reported, because their events will have no returns.

In [19]:
# 3b. For each company (CIK), pick the linked index ticker with price data and the latest membership end date.
tickers_with_prices = set(df_weekly["ticker"])
latest_end = df_historical_clean.groupby("ticker")["valid_to"].max().rename("latest_end").reset_index()
price_choice = (df_ticker_cik[["issuer_cik", "ticker"]].merge(latest_end, on="ticker")
                .loc[lambda d: d["ticker"].isin(tickers_with_prices)]
                .sort_values("latest_end").drop_duplicates("issuer_cik", keep="last")
                .rename(columns={"ticker": "price_ticker"})[["issuer_cik", "price_ticker"]])

renamed = (df_sp_trades[["issuer_cik", "sp_ticker"]].drop_duplicates()
           .merge(price_choice, on="issuer_cik").query("sp_ticker != price_ticker"))
print(f"Companies with a usable price series: {price_choice['issuer_cik'].nunique()} of {df_sp_trades['issuer_cik'].nunique()}")
print("Examples where an old index ticker uses its newer ticker's prices:")
print(renamed.drop_duplicates("sp_ticker").head(10).to_string(index=False))

no_prices = df_sp_trades[~df_sp_trades["issuer_cik"].isin(price_choice["issuer_cik"])]
print(f"\nTrades in companies with no price series: {len(no_prices):,} "
      f"({no_prices['trans_code'].eq('P').sum():,} purchases) in {no_prices['issuer_cik'].nunique()} companies, e.g.",
      sorted(no_prices["sp_ticker"].unique())[:12])

Companies with a usable price series: 540 of 593
Examples where an old index ticker uses its newer ticker's prices:
issuer_cik sp_ticker price_ticker
0001326801        FB         META
0001140859       ABC          COR
0000009389       BLL         BALL
0001156039      ANTM          ELV
0001095073        RE           EG
0000031791       PKI         RVTY
0000101829       UTX          RTX
0001175454       FLT         CPAY
0000765880      PEAK          DOC
0000813828      VIAC         PARA

Trades in companies with no price series: 4,511 (271 purchases) in 53 companies, e.g. ['ABMD', 'ADS', 'ALXN', 'ANSS', 'ATVI', 'AVB', 'BK', 'CDAY', 'CERN', 'CMA', 'COG', 'CTLT']


### Step 4: Building the event table and measuring abnormal returns

This step turns individual trades into the **analysis-ready dataset**. Each row is one *event*: a week in which insiders at one company bought (or sold), together with what the stock did afterward compared with what the market alone would predict.

#### 4a. Roll trades up into company-week events

**Why not one row per trade?** An investor reading EDGAR asks one question per company per week: *did insiders buy here?* If a CEO and two directors all buy Boeing in the same week, that is one signal, not three, and counting it three times would let one busy company dominate the averages.

**Event week.** The SEC posts a filing to its public index the business day after it is received, so information becomes tradeable on *filing date + 1 business day*. The event week is the week ending on the first **Friday on or after** that date, and we assume an investor buys at that Friday's close. This rule never uses a price from before the public could know.

For each company, event week, and direction (P or S), we record:

- `n_trades` and `n_insiders`: how many trades and distinct insiders.
- `total_value`: the total dollar value traded, and `log_value`, its natural log. Trade sizes range from hundreds of dollars to hundreds of millions, so the log compresses that range for regression (as in Lectures 8 to 10).
- `role`: the **most senior** role among the traders that week, plus `any_c_suite` (1 if any C-suite officer traded).
- `any_plan_10b5_1`: 1 if any trade was marked as made under a pre-scheduled plan (available from April 2023).
- `mixed_week`: 1 if the same company had both insider buying and selling that week.

In [ ]:
# 4a. Assign each trade to its event week and aggregate to one row per company, week, and direction.
df_sp_trades["public_date"] = df_sp_trades["filing_date"] + pd.offsets.BDay(1)
# First Friday on or after the public date (rollforward leaves a date that is already a Friday unchanged).
df_sp_trades["event_week"] = df_sp_trades["public_date"].map(pd.offsets.Week(weekday=4).rollforward)

df_events = (df_sp_trades.groupby(["issuer_cik", "event_week", "trans_code"])
             .agg(issuer_name=("issuer_name", "first"), sp_ticker=("sp_ticker", "first"),
                  n_trades=("accession", "size"), n_insiders=("owner_cik", "nunique"),
                  total_value=("trade_value", "sum"), role_rank=("role_rank", "min"),
                  any_c_suite=("role_rank", lambda r: int((r == 1).any())),
                  any_plan_10b5_1=("plan_10b5_1", "max"), mean_filing_lag=("filing_lag_days", "mean"),
                  first_filing=("filing_date", "min"))
             .reset_index())
df_events["role"] = df_events["role_rank"].map(role_names)
df_events["log_value"] = np.log(df_events["total_value"])
both = df_events.groupby(["issuer_cik", "event_week"])["trans_code"].transform("nunique")
df_events["mixed_week"] = both.gt(1).astype(int)
df_events = df_events.merge(price_choice, on="issuer_cik", how="left")
print(df_events.groupby("trans_code").agg(events=("issuer_cik", "size"), companies=("issuer_cik", "nunique"),
                                          trades=("n_trades", "sum")).rename(index={"P": "purchases", "S": "sales"}).to_string())

#### 4b. Expected returns from the market model

Stocks rise and fall with the market, so a stock going up after an insider buy proves nothing on its own. We need each stock's **expected** return: what it would normally do, given what the market did.

The **market model** describes a stock's weekly return as

$$r_{stock} = \alpha + \beta \cdot r_{market} + \varepsilon$$

- **β (beta)** measures how strongly the stock moves with the market. A β of 1.5 means that when SPY rises 1%, the stock typically rises 1.5%.
- **α (alpha)** is the stock's typical weekly return when the market is flat.
- **ε** is the part the market does not explain.

This is the single-variable linear regression from Lecture 7. For each event we fit it on the stock's **52 weekly returns before the event week** (the *estimation window*), requiring at least 40 weeks of data. Using only pre-event weeks matters: if post-event weeks were included, an insider's good news would leak into "normal" behavior and shrink the abnormal return we are trying to measure.

For a single input, the least-squares slope equals cov(stock, market) / var(market), and the intercept equals mean(stock) − β × mean(market). We use those closed-form formulas because they are exactly what `LinearRegression().fit()` returns, and are much faster across 30,000+ regressions.

In [ ]:
# 4b. Estimate alpha and beta for each event from the 52 weeks before its event week.
weekly_by_ticker = {t: g.set_index("week")[["ret", "mkt_ret"]].sort_index() for t, g in df_weekly.groupby("ticker")}
ESTIMATION_WEEKS, MIN_WEEKS = 52, 40

def market_model(ticker, event_week):
    """Return (alpha, beta, weeks used) from the 52 weeks strictly before event_week."""
    series = weekly_by_ticker.get(ticker)
    if series is None:
        return np.nan, np.nan, 0
    window = series.loc[:event_week - pd.Timedelta(days=1)].tail(ESTIMATION_WEEKS)
    if len(window) < MIN_WEEKS:
        return np.nan, np.nan, len(window)
    beta = np.cov(window["ret"], window["mkt_ret"])[0, 1] / window["mkt_ret"].var()
    alpha = window["ret"].mean() - beta * window["mkt_ret"].mean()
    return alpha, beta, len(window)

fits = [market_model(t, w) for t, w in zip(df_events["price_ticker"], df_events["event_week"])]
df_events[["alpha", "beta", "estimation_weeks"]] = pd.DataFrame(fits, index=df_events.index)
print(f"Events with an estimated beta: {df_events['beta'].notna().sum():,} of {len(df_events):,}")
print(df_events["beta"].describe(percentiles=[.05, .5, .95]).round(2).to_string())

#### 4c. Abnormal and cumulative abnormal returns

For every week *after* the event week:

- **Abnormal return (AR)** = actual return − expected return = $r_{stock} - (\hat\alpha + \hat\beta \cdot r_{market})$
- **Cumulative abnormal return (CAR)** over *h* weeks = the sum of the weekly ARs from week 1 to week *h*.

A CAR of **+5%** at 13 weeks means the stock did 5 percentage points better over the following quarter than the market model predicted. We compute CAR at **1, 4, 13, and 26 weeks**, our short-to-medium horizons. A CAR is left missing when the stock lacks a full window of prices (for example, events in late 2025 whose 26-week window runs past our data).

We also compute **`car_pre4`**, the CAR over the 4 weeks *ending with* the event week. It is a diagnostic, not a result: it tells us whether prices were already moving before the public could know (for example, if insiders tend to buy after a price drop).

In [ ]:
# 4c. Abnormal returns: sum (actual - expected) over 1, 4, 13, 26 weeks after the event week, plus the 4 weeks ending with it.
HORIZONS = [1, 4, 13, 26]

def cumulative_abnormal_returns(ticker, event_week, alpha, beta):
    series = weekly_by_ticker.get(ticker)
    out = {f"car_{h}w": np.nan for h in HORIZONS} | {"car_pre4": np.nan}
    if series is None or np.isnan(beta):
        return out
    abnormal = series["ret"] - (alpha + beta * series["mkt_ret"])
    after = abnormal.loc[event_week + pd.Timedelta(days=1):]
    for h in HORIZONS:
        window = after.iloc[:h]
        # Require h consecutive weekly returns that end within h weeks plus a holiday buffer of the event week.
        if len(window) == h and (window.index[-1] - event_week).days <= 7 * h + 7:
            out[f"car_{h}w"] = window.sum()
    before = abnormal.loc[:event_week].tail(4)
    if len(before) == 4:
        out["car_pre4"] = before.sum()
    return out

cars = [cumulative_abnormal_returns(t, w, a, b) for t, w, a, b in
        zip(df_events["price_ticker"], df_events["event_week"], df_events["alpha"], df_events["beta"])]
df_events = pd.concat([df_events, pd.DataFrame(cars, index=df_events.index)], axis=1)
car_cols = [f"car_{h}w" for h in HORIZONS]
print("Events with each CAR available:")
print(df_events.groupby("trans_code")[car_cols + ["car_pre4"]].count().rename(index={"P": "purchases", "S": "sales"}).to_string())

#### 4d. Save the analysis-ready dataset

The final file, `data/processed/insider_events.csv`, has one row per *(company, event week, direction)* and is the dataset described in the **Data description** section. Events without a usable price series or beta are kept in the file, so the sample-loss counts stay reproducible, but they are excluded from return analyses.

The cleaning log below summarizes the whole pipeline, from raw transactions to events.

In [ ]:
# 4d. Order columns, save the analysis-ready event table, and print the full cleaning funnel.
event_cols = ["issuer_cik", "issuer_name", "sp_ticker", "price_ticker", "trans_code", "event_week", "first_filing",
              "n_trades", "n_insiders", "total_value", "log_value", "role", "role_rank", "any_c_suite",
              "any_plan_10b5_1", "mixed_week", "mean_filing_lag", "alpha", "beta", "estimation_weeks",
              "car_pre4"] + car_cols
df_events = df_events[event_cols].sort_values(["event_week", "issuer_cik", "trans_code"]).reset_index(drop=True)
df_events.to_csv(processed_dir / "insider_events.csv", index=False)

has_returns = df_events["beta"].notna()
cleaning_log.append({"step": "4a. Trades rolled up into company-week events (rows = events)", "rows": len(df_events)})
cleaning_log.append({"step": "4b. Events with a price series and >= 40 weeks for beta", "rows": int(has_returns.sum())})
df_cleaning_log = pd.DataFrame(cleaning_log)
display(df_cleaning_log)
print(f"Saved {len(df_events):,} events ({df_events['trans_code'].eq('P').sum():,} purchase events, "
      f"{df_events['trans_code'].eq('S').sum():,} sale events) to data/processed/insider_events.csv")
display(df_events[df_events["trans_code"].eq("P")].head())

### Saving intermediate and cleaned data

The cleaning cells save the following files for reuse:

- **Step 1:** `data/interim/sec_ps_trades.csv.gz`, cleaned SEC purchase and sale transactions.
- **Step 2:** `data/interim/sp500_trades.csv.gz`, S&P 500 trades with insider roles, and `data/interim/sp500_ticker_cik_links.csv`, ticker-to-company links.
- **Membership cleaning:** `data/processed/historical_membership_clean.csv`, continuous membership periods grouped by ticker.
- **Step 4:** `data/processed/insider_events.csv`, the analysis-ready company-week-direction event table, written when Step 4 is run.

**Execution status:** The four Step 4 code cells (4a to 4d) need a fresh run. Their outputs and the event CSV have not been generated in this editing session. The handoff reports prior full-data results of **27,820 events** (2,356 purchase and 25,464 sale events), **26,100 estimated betas**, and CARs for **2,227 purchase and 23,873 sale events** at each horizon. These are reference counts, not newly verified outputs.

CSV files do not retain date types or leading zeros in identifiers automatically. When reloading, preserve CIKs as strings and parse date columns explicitly, including `snapshot_date`, `valid_from`, and `valid_to` for membership and `event_week` and `first_filing` for events.

# __EXPLORATORY DATA ANALYSIS__

### Summary statistics

[Report sample size, company counts, date coverage, missingness, purchase/sale
counts, and typical trade sizes. Explain what each summary tells us.]

In [20]:
# Add summary statistics using the real cleaned dataset.

### Plots

Possible initial plots include purchases versus sales, trade-size distributions,
filing counts over time, and the gap between trade dates and filing dates.

[Choose plots based on the actual data. Put a brief explanation before each plot
and an interpretation after it. Use Seaborn and NumPy/SciPy where useful.]

Do not run the preregistered hypothesis tests before the preregistrations are finalized.

In [21]:
# Add focused Seaborn plots after the data has been cleaned.

### What we learned from the EDA

[Describe findings using actual numbers and plots. Explain whether the research
questions or dataset description should change based on what the EDA reveals.]

# __DATA DESCRIPTION__

### Motivation

**Why was this dataset created?**

The combined dataset will let us examine whether public insider disclosures
predict stock returns beyond what would normally be expected from market movements.
The SEC source records regulatory disclosures; the price source records market outcomes.

### Composition

**What do the observations and attributes represent?**

[Describe the actual analysis-ready rows and columns. Distinguish transaction rows
from any company/disclosure-day events created for analysis. Add a data dictionary.]

**How many observations are included, and what period do they cover?**

[Complete after cleaning.]

### Collection and preprocessing

**How was the combined dataset created?**

[Summarize the sources, tools, selection criteria, cleaning, joins, and transformations
actually used. Refer to the cleaning appendix.]

**What influenced which records were observed?**

[Explain reporting requirements, company selection, and price availability.]

**If people are represented, what was the purpose of the original collection?**

The SEC records concern corporate insiders whose transactions and holdings are
subject to regulatory disclosure. Filing these records does not establish consent
to every downstream interpretation. Avoid inferring misconduct from an association.

**Where can the original data be found?**

[Add source links and the course-approved shared raw-data location.]

# __DATA LIMITATIONS__

Potential limitations to evaluate with the actual data:

1. **Membership coverage and sample selection:** historical membership can reduce
   selection based on current constituents, but the community list may contain errors
   and still requires date/symbol validation. An S&P 500 sample mainly represents large firms.
2. **Missing and changing symbols:** a historical ticker may not match the vendor's
   symbol, and delisted or acquired companies may lack complete histories.
3. **Disclosure timing:** transaction dates differ from public disclosure dates;
   event windows must use information available at the assumed trading time.
4. **Amendments and joint filings:** corrections and multiple reporting owners can
   create duplicates or incorrect attribution if handled carelessly.
5. **Interpretation:** sales may reflect personal liquidity or diversification.
   Earnings and other news can explain returns. Associations do not establish causation.
6. **People and impact:** results should not label individuals as engaging in illegal
   activity or be presented as guaranteed trading profits.

[Add measured data gaps and explain how each relevant limitation changes what
the project can conclude.]

# __PREREGISTRATION STATEMENT 1__

**Draft hypothesis:** Public insider purchase disclosures are followed by positive
average short-horizon cumulative abnormal returns.

**Proposed analysis:** [Before running this test, specify the event unit, sample,
disclosure/entry rule, return horizon, market-model estimation window, exclusions,
statistical test/model, uncertainty calculation, and significance threshold.]

**Status:** Draft, not tested. Finalize and record this statement before examining
the corresponding outcome relationship. Report the final result regardless of significance.

# __PREREGISTRATION STATEMENT 2__

**Draft hypothesis:** C-suite purchase disclosures are followed by larger abnormal
returns than purchase disclosures by other insiders.

**Proposed analysis:** [Before running this test, define role categories and the
comparison sample, specify a feasible regression/model and controls, and address
joint filings, repeated companies, and uncertainty.]

**Status:** Draft, not tested. Confirm that the actual data supports this comparison,
then finalize the analysis before examining the corresponding outcome relationship.
At least one final-project model must address a research question.

# __QUESTIONS FOR REVIEWERS__

1. Does our actual sample support the proposed comparison between C-suite insiders
   and other insiders?
2. Is our event-date and trading-entry rule appropriate for the disclosure and price
   data available?
3. Are our amendment handling, missing-price rules, and company-selection decisions reasonable?
4. What changes would make the preregistered models feasible and informative using
   course methods?

[Update these questions after the group has discussed the real EDA.]